# Demo 2 — Một outlier có thể phá decision boundary

**Slide:** 6 · **Thời lượng:** 30–60 giây · **Hình:** `02_edge_case_outlier.png`

## Ý chính

- **Outlier** = mẫu khác biệt về **dữ liệu**.
- **Edge case** = mẫu làm model **tệ đi đáng kể về hiệu năng**.

Một outlier nhãn sai có thể kéo lệch boundary và làm model sai hàng loạt trên dữ liệu sạch.

> **Câu chốt:** outlier nói về dữ liệu; edge case nói về hiệu năng.

## Bước 0 — Thiết lập

In [ ]:
# --- Thiết lập: tìm và import helper từ ../code-demo ---
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

for candidate in [
    Path.cwd() / "code-demo",
    Path.cwd().parent / "code-demo",
    Path.cwd().parent.parent / "code-demo",
]:
    if (candidate / "common.py").exists():
        CODE_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError("Không tìm thấy code-demo/common.py")

sys.path.insert(0, str(CODE_DIR))

from common import accuracy, predict, save, setup, train_logreg  # noqa: E402

setup()
print("code-demo:", CODE_DIR)
print("Hình sẽ lưu vào:", CODE_DIR / "figures")

## Bước 1 — Tạo dữ liệu sạch và thêm đúng một outlier

- 10 mẫu mỗi lớp để một outlier có đủ trọng lượng.
- Outlier nằm sâu trong vùng lớp 1 nhưng bị gán nhãn 0.

In [ ]:
rng = np.random.default_rng(7)


def make_data(rng, n=200):
    X0 = rng.normal([-1.0, -1.0], 0.7, (n, 2))
    X1 = rng.normal([1.0, 1.0], 0.7, (n, 2))
    X = np.vstack([X0, X1])
    y = np.r_[np.zeros(n), np.ones(n)].astype(int)
    return X, y


X, y = make_data(rng, n=10)

outlier = np.array([[5.0, 5.0]])
y_out = np.array([0])

X_dirty = np.vstack([X, outlier])
y_dirty = np.r_[y, y_out]

print("Train sạch :", X.shape)
print("Train bẩn  :", X_dirty.shape, "— thêm 1 outlier nhãn 0 tại", outlier[0])

## Bước 2 — Train hai model và đánh giá trên tập test sạch

Cùng dữ liệu test, chỉ khác tập train có/không có outlier.

In [ ]:
w_clean = train_logreg(X, y, lr=0.2, epochs=2000, l2=1e-4)
w_dirty = train_logreg(X_dirty, y_dirty, lr=0.2, epochs=2000, l2=1e-4)

X_test, y_test = make_data(np.random.default_rng(99))
acc_clean = accuracy(y_test, predict(X_test, w_clean))
acc_dirty = accuracy(y_test, predict(X_test, w_dirty))

print("=== DEMO 2: một outlier có thể phá decision boundary ===")
print(f"Accuracy trên test sạch — không outlier : {acc_clean:.3f}")
print(f"Accuracy trên test sạch — có 1 outlier : {acc_dirty:.3f}  ← giảm {acc_clean - acc_dirty:.3f}")
print()
print("Outlier  = mẫu khác biệt về DỮ LIỆU.")
print("Edge case= mẫu model làm TỆ ĐI ĐÁNG KỂ về HIỆU NĂNG.")
print("Outlier có thể gây edge case, nhưng không phải outlier nào cũng vậy.")

## Bước 3 — Vẽ decision boundary trước và sau

Đường đen là boundary. Chữ X đỏ là outlier nhãn sai.

In [ ]:
def boundary_line(w, xlim):
    """w = [b, w1, w2]; vẽ w1*x + w2*y + b = 0."""
    xs = np.linspace(*xlim, 200)
    if abs(w[2]) < 1e-9:
        return xs, np.full_like(xs, np.nan)
    return xs, -(w[0] + w[1] * xs) / w[2]


fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
xlim = (-3.5, 6.5)
ylim = (-3.5, 6.5)

for ax, (w, title, acc) in zip(
    axes,
    [
        (w_clean, "Không có outlier", acc_clean),
        (w_dirty, "Có 1 outlier nhãn sai", acc_dirty),
    ],
):
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=14, alpha=0.55, label="Lớp 0", color="#1f77b4")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=14, alpha=0.55, label="Lớp 1", color="#ff7f0e")
    xs, ys = boundary_line(w, xlim)
    ax.plot(xs, ys, "k-", lw=2, label="Decision boundary")
    if "outlier" in title:
        ax.scatter(*outlier[0], s=220, marker="X", color="#d62728", edgecolor="k", zorder=5, label="Outlier (nhãn 0)")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_title(f"{title}\nAccuracy test = {acc:.2f}")
    ax.set_xlabel("x₁")

axes[0].set_ylabel("x₂")
axes[0].legend(loc="upper left", fontsize=8)

save(fig, "02_edge_case_outlier.png")
plt.show()

## Đọc kết quả & ý nghĩa cho slide

| Trường hợp | Accuracy test sạch |
|---|---|
| Không outlier | **~0.99** |
| Thêm 1 outlier nhãn sai | **~0.82** |

**Nói trên slide:**

> “Bên trái: accuracy 0,99. Bên phải: thêm đúng một outlier nhãn sai, accuracy còn 0,82. Một mẫu cực đoan đủ để kéo boundary và làm sai hàng loạt.”

**Liên hệ với 99,99% an toàn:**

> 1 triệu chuyến → 100 chuyến lỗi. Nếu 0,01% đó gây tai nạn chết người, bạn có dùng xe tự lái không?

**Câu chốt:** outlier nói về dữ liệu; edge case nói về hiệu năng.